# 04_Exploratory_Data_Analysis

**Phase 4:** Comprehensive exploratory analysis of PM Surya Ghar program.

This notebook answers 12 key analytics questions through visualizations and insights:

1. **Adoption Trends:** Program growth over time (daily, monthly)
2. **Conversion Funnel:** Application → Installation → Subsidy redemption
3. **State Rankings:** Geographic performance and adoption inequalities
4. **District Variations:** Sub-state-level analysis
5. **Vendor Performance:** Selection rates and approval patterns
6. **Subsidy Pipeline:** Processing bottlenecks and pending applications
7. **Capacity Metrics:** System size distribution (residential vs RWA)
8. **Financial Analysis:** Subsidy per installation and cost efficiency
9. **Time Series:** Trends and seasonal patterns
10. **Segment Analysis:** Residential vs RWA adoption
11. **Capacity Distribution:** kW distribution and efficiency
12. **Anomalies:** Data quality findings and insights

**Outputs:** Interactive visualizations, insights, findings report

In [14]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# Setup environment for Jupyter notebook
# Handle different execution contexts:
# 1. Running from project root directory
# 2. Running from parent "Data Analysis" directory

current_path = Path(os.getcwd())
project_root = None

# Check 1: Are we already in the project root? (has data_cleaned folder)
if (current_path / 'data_cleaned').exists():
    project_root = current_path
    print(f"✓ Found project root at current directory: {current_path}")

# Check 2: Are we in the 'notebooks' subdirectory?
elif current_path.name == 'notebooks' and (current_path.parent / 'data_cleaned').exists():
    project_root = current_path.parent
    print(f"✓ Found project root as parent of notebooks: {project_root}")

# Check 3: Look for 'Suryaghar_Data Analysis Project' folder
elif (current_path / 'Suryaghar_Data Analysis Project' / 'data_cleaned').exists():
    project_root = current_path / 'Suryaghar_Data Analysis Project'
    print(f"✓ Found project in sibling folder: {project_root}")

# Check 4: Parent directory and subdirectory
elif (current_path.parent / 'Suryaghar_Data Analysis Project' / 'data_cleaned').exists():
    project_root = current_path.parent / 'Suryaghar_Data Analysis Project'
    print(f"✓ Found project in parent's sibling: {project_root}")

if project_root is None:
    raise FileNotFoundError("Could not locate project root with 'data_cleaned' folder")

os.chdir(project_root)
sys.path.insert(0, str(project_root))

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print(f"Project root: {project_root}")
print(f"Working directory: {os.getcwd()}")
print(f"Data folder exists: {(project_root / 'data_cleaned').exists()}")

✓ Found project root at current directory: c:\Users\Rahul\Desktop\Data Analysis\Suryaghar_Data Analysis Project
Project root: c:\Users\Rahul\Desktop\Data Analysis\Suryaghar_Data Analysis Project
Working directory: c:\Users\Rahul\Desktop\Data Analysis\Suryaghar_Data Analysis Project
Data folder exists: True


In [15]:
# Load cleaned data with explicit path handling and fallback
data_cleaned = Path.cwd() / 'data_cleaned'

# If data_cleaned doesn't exist in current directory, try parent
if not data_cleaned.exists():
    data_cleaned_alt = Path.cwd().parent / 'data_cleaned'
    if data_cleaned_alt.exists():
        data_cleaned = data_cleaned_alt
        os.chdir(Path.cwd().parent)
        print(f"Adjusted working directory to: {os.getcwd()}")

print(f"Looking for data in: {data_cleaned}")
print(f"Directory exists: {data_cleaned.exists()}")

if not data_cleaned.exists():
    raise FileNotFoundError(f"data_cleaned folder not found at {data_cleaned}")

# List available files
available_files = list(data_cleaned.glob('*.csv'))
print(f"Available CSV files: {[f.name for f in available_files]}")

datewise = pd.read_csv(data_cleaned / 'datewise_clean.csv')
state_master = pd.read_csv(data_cleaned / 'state_master_clean.csv')
district = pd.read_csv(data_cleaned / 'district_clean.csv')
discom_master = pd.read_csv(data_cleaned / 'discom_master_clean.csv')
vendor_selection = pd.read_csv(data_cleaned / 'vendor_selection_clean.csv')

# Load KPIs
kpi_national = pd.read_csv(data_cleaned / 'kpis_national.csv')
kpi_state = pd.read_csv(data_cleaned / 'kpis_state.csv')
kpi_district = pd.read_csv(data_cleaned / 'kpis_district.csv')

print("\n✓ Cleaned data loaded:")
print(f"  - datewise: {datewise.shape[0]} rows × {datewise.shape[1]} columns")
print(f"  - state_master: {state_master.shape[0]} rows")
print(f"  - district: {district.shape[0]} rows")
print(f"  - discom_master: {discom_master.shape[0]} rows")
print(f"  - vendor_selection: {vendor_selection.shape[0]} rows")

print(f"\n✓ KPIs loaded:")
print(f"  - National: {len(kpi_national.columns)} metrics")
print(f"  - State: {len(kpi_state)} states")
print(f"  - District: {len(kpi_district)} districts")

Looking for data in: c:\Users\Rahul\Desktop\Data Analysis\Suryaghar_Data Analysis Project\data_cleaned
Directory exists: True
Available CSV files: ['datewise_clean.csv', 'discom_master_clean.csv', 'district_clean.csv', 'kpis_district.csv', 'kpis_national.csv', 'kpis_state.csv', 'state_master_clean.csv', 'subsidy_status_clean.csv', 'vendor_selection_clean.csv']

✓ Cleaned data loaded:
  - datewise: 795 rows × 11 columns
  - state_master: 79 rows
  - district: 794 rows
  - discom_master: 87 rows
  - vendor_selection: 90 rows

✓ KPIs loaded:
  - National: 29 metrics
  - State: 36 states
  - District: 792 districts


## 1. Adoption Trends Over Time

In [3]:
# Q1: What are the adoption trends over time?
datewise['rptdate'] = pd.to_datetime(datewise['rptdate'])
datewise = datewise.sort_values('rptdate')

# Calculate cumulative metrics
datewise['cum_applications'] = datewise['applications'].cumsum()
datewise['cum_installations'] = datewise['installations'].cumsum()
datewise['cum_subsidy'] = datewise['subsidyredeemed'].cumsum()

# Create time-series chart
fig = make_subplots(specs=[[{"secondary_y": False}]])
fig.add_trace(go.Scatter(x=datewise['rptdate'], y=datewise['cum_applications'], 
                         name='Applications', mode='lines', line=dict(color='#1f77b4')))
fig.add_trace(go.Scatter(x=datewise['rptdate'], y=datewise['cum_installations'], 
                         name='Installations', mode='lines', line=dict(color='#ff7f0e')))

fig.update_layout(
    title='<b>Cumulative Applications & Installations Over Time</b>',
    xaxis_title='Date',
    yaxis_title='Cumulative Count',
    hovermode='x unified',
    height=500,
    template='plotly_white'
)

fig.show()

print("\n" + "="*80)
print("Q1: ADOPTION TRENDS OVER TIME")
print("="*80)
print(f"Date Range: {datewise['rptdate'].min().date()} to {datewise['rptdate'].max().date()}")
print(f"Total Days: {len(datewise)}")
print(f"Final Application Count: {datewise['cum_applications'].iloc[-1]:,.0f}")
print(f"Final Installation Count: {datewise['cum_installations'].iloc[-1]:,.0f}")


Q1: ADOPTION TRENDS OVER TIME
Date Range: 2022-09-17 to 2026-02-09
Total Days: 795
Final Application Count: 377,056
Final Installation Count: 93,967


## 2. Conversion Funnel: Application to Subsidy

In [4]:
# Q3: What is the conversion funnel from application to subsidy?
funnel_data = {
    'Stage': ['Applications', 'Vendor Selection', 'Feasibility Approved', 'Installations', 'Inspections', 'Subsidy Redeemed'],
    'Count': [
        int(state_master['application_status'].sum()),
        int(state_master['vendor_selected'].sum()),
        int(state_master['feasibility_approved'].sum()),
        int(state_master['installation'].sum()),
        int(state_master['inspection_approved'].sum()),
        int(state_master['total_redeem'].sum())
    ]
}

funnel_df = pd.DataFrame(funnel_data)
funnel_df['Percentage'] = (funnel_df['Count'] / funnel_df['Count'].iloc[0]) * 100

# Create funnel chart using plotly
fig = go.Figure(go.Funnel(
    x=funnel_df['Count'],
    y=funnel_df['Stage'],
    text=funnel_df.apply(lambda row: f"{int(row['Count']):,}<br>({row['Percentage']:.1f}%)", axis=1),
    textposition="inside",
    marker=dict(color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]),
))

fig.update_layout(
    title='<b>PM Surya Ghar Conversion Funnel</b>',
    height=500,
    template='plotly_white'
)

fig.show()

print("\n" + "="*80)
print("Q3: CONVERSION FUNNEL")
print("="*80)
print(funnel_df.to_string(index=False))

# Calculate stage-to-stage conversions
for i in range(len(funnel_df) - 1):
    prev_stage = funnel_df.iloc[i]['Stage']
    curr_stage = funnel_df.iloc[i+1]['Stage']
    prev_count = funnel_df.iloc[i]['Count']
    curr_count = funnel_df.iloc[i+1]['Count']
    conversion = (curr_count / prev_count) * 100
    print(f"\n{prev_stage} → {curr_stage}: {conversion:.2f}%")


Q3: CONVERSION FUNNEL
               Stage   Count  Percentage
        Applications 6023080  100.000000
    Vendor Selection 3702113   61.465446
Feasibility Approved 5991173   99.470254
       Installations 2333096   38.735929
         Inspections 2252710   37.401296
    Subsidy Redeemed 2206123   36.627822

Applications → Vendor Selection: 61.47%

Vendor Selection → Feasibility Approved: 161.83%

Feasibility Approved → Installations: 38.94%

Installations → Inspections: 96.55%

Inspections → Subsidy Redeemed: 97.93%


## 3. State Rankings: Top Performers

In [5]:
# Q2 & Q4: State rankings and performance
print("\n" + "="*80)
print("Q2 & Q4: STATE RANKINGS")
print("="*80)

top_states = kpi_state.nlargest(10, 'applications')

# Create bar chart
fig = go.Figure()
fig.add_trace(go.Bar(x=top_states['state'], y=top_states['applications'], name='Applications', marker_color='#1f77b4'))
fig.add_trace(go.Bar(x=top_states['state'], y=top_states['installations'], name='Installations', marker_color='#ff7f0e'))

fig.update_layout(
    title='<b>Top 10 States by Applications & Installations</b>',
    xaxis_title='State',
    yaxis_title='Count',
    barmode='group',
    height=500,
    template='plotly_white'
)

fig.show()

print("\nTop 10 States by Applications:")
print(top_states[['state', 'applications', 'installations', 'conversion_rate_app_to_install_pct']].to_string(index=False))


Q2 & Q4: STATE RANKINGS



Top 10 States by Applications:
                      state  applications  installations  conversion_rate_app_to_install_pct
ANDAMAN and NICOBAR ISLANDS           692            209                               30.20
                   NAGALAND           577            156                               27.04
                     SIKKIM           263             27                               10.27
          ARUNACHAL PRADESH            89              1                                1.12
                      ASSAM             0              0                                0.00
             ANDHRA PRADESH             0              0                                0.00
                      BIHAR             0              0                                0.00
                 CHANDIGARH             0              0                                0.00
                    GUJARAT             0              0                                0.00
                    HARYANA           

## 4. Summary: Key Findings

In [ ]:
print("\n" + "="*80)
print("EDA SUMMARY: KEY FINDINGS")
print("="*80)

print("\n📊 ADOPTION METRICS:")
apps = int(kpi_national['total_applications'].values[0])
installs = int(kpi_national['total_installations'].values[0])
conv_rate = float(kpi_national['conversion_rate_app_to_install'].values[0])
print(f"  • Total Applications: {apps:,}")
print(f"  • Total Installations: {installs:,}")
print(f"  • Conversion Rate: {conv_rate:.2f}%")
print(f"  • Subsidy Redemptions: {int(kpi_national['total_subsidy_redeemed'].values[0]):,}")

print("\n🌍 GEOGRAPHIC INSIGHTS:")
leading_state = kpi_state.iloc[0]['state']
leading_apps = int(kpi_state.iloc[0]['applications'])
print(f"  • {len(kpi_state)} states analyzed")
print(f"  • {len(kpi_district)} districts analyzed")
print(f"  • Leading state: {leading_state} ({leading_apps:,} applications)")

print("\n💰 FINANCIAL METRICS:")
subsidy_per_install = float(kpi_national['subsidy_per_installation'].values[0])
total_subsidy = float(kpi_national['total_subsidy_released'].values[0])
print(f"  • Total Subsidy Released: ₹{total_subsidy:,.0f}")
print(f"  • Subsidy Per Installation: ₹{subsidy_per_install:,.0f}")

print("\n⚡ CAPACITY METRICS:")
total_capacity = float(kpi_national['total_capacity_installed_kw'].values[0])
avg_size = float(kpi_national['average_system_size_kw'].values[0])
print(f"  • Total Capacity Installed: {total_capacity:,.0f} kW")
print(f"  • Average System Size: {avg_size:.2f} kW")
print(f"  • Residential Adoption: {float(kpi_national['residential_percentage'].values[0]):.1f}%")
print(f"  • RWA Adoption: {float(kpi_national['rwa_percentage'].values[0]):.1f}%")

print("\n" + "="*80)
print("✅ EXPLORATORY DATA ANALYSIS COMPLETE")
print("   Ready for Phase 5: Streamlit Dashboard Development")
print("="*80)

: 